In [ ]:
%config InlineBackend.figure_format = 'svg'

import numpy as np
import matplotlib.pyplot as plt

m_L = 80000.0
m_C = 40000.0
b_CL = 50000.0
b_L = 5000.0
b_C = 3000.0
k_CL = 1250000.0 / 3.0
v_cruise = 100.0 / 3.6

A = np.array([
    [-(b_CL + b_L)/m_L,  b_CL/m_L,         -k_CL/m_L],
    [ b_CL/m_C,         -(b_CL + b_C)/m_C,  k_CL/m_C],
    [ 1.0,              -1.0,               0.0     ]
])

B = np.array([
    [1/m_L,  1/m_L],
    [0.0,   -1/m_C],
    [0.0,    0.0  ]
])

C = np.array([[0.0, 0.0, 1.0]])
D = np.array([[0.0, 0.0]])

poles = np.linalg.eigvals(A)
z_L = -b_C / m_C

print('Poles:', poles)
print(f'Zero G_L (FL -> delta): {z_L:.4f} rad/s')

plt.figure(figsize=(6, 4))
plt.plot(np.real(poles), np.imag(poles), 'rx', markersize=8, markeredgewidth=2, label='Poles')
plt.plot(z_L, 0, 'bo', markersize=8, fillstyle='none', markeredgewidth=2, label=f'Zero ({z_L:.4f} rad/s)')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Real')
plt.ylabel('Imaginary')
plt.title('Pole-Zero Plot')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('PoleZeroPlot.pdf')
plt.show()

x = np.array([0.0, 0.0, 0.0])
u = np.array([0.0, 0.0])
u_perturb = np.array([50000.0, 0.0])

dt = 0.001
t_end = 100
time = np.arange(0, t_end, dt)

def sys_deriv(x_state, t_val):
    u_val = u_perturb if t_val >= 0 else u
    return np.dot(A, x_state) + np.dot(B, u_val)

def solve_rk4(x0, time, dt):
    x_curr = x0.copy()
    hist = np.zeros((len(time), 3))
    for i, t in enumerate(time):
        hist[i] = x_curr
        k1 = sys_deriv(x_curr, t)
        k2 = sys_deriv(x_curr + 0.5 * dt * k1, t + 0.5 * dt)
        k3 = sys_deriv(x_curr + 0.5 * dt * k2, t + 0.5 * dt)
        k4 = sys_deriv(x_curr + dt * k3, t + dt)
        x_curr = x_curr + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
    return hist

x_hist = solve_rk4(x, time, dt)
y_hist = (C @ x_hist.T).flatten()
x_hist = x_hist + [v_cruise, v_cruise, 0]

delta_peak = np.max(y_hist)
idx_peak = np.argmax(y_hist)
delta_ss = y_hist[-1]
v_ss = x_hist[-1, 0]

plt.figure(figsize=(8, 4))
plt.plot(time, x_hist[:, 0], label=r'Locomotive Velocity ($v_L$)', color='red')
plt.plot(time, x_hist[:, 1], label=r'Cabin Velocity ($v_C$)', color='green', linestyle='--')
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.title('Velocities')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Velocities.pdf')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time, x_hist[:, 0], label=r'Locomotive Velocity ($v_L$)', color='red')
plt.plot(time, x_hist[:, 1], label=r'Cabin Velocity ($v_C$)', color='green', linestyle='--')
plt.ylim(27.5, 29)
plt.xlim(0, 4)
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.title('Velocities (Zoomed 4s)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('VelocitiesZ.pdf')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time, y_hist, label=r'Coupler Deflection ($\delta$)', color='blue')
plt.plot(time[idx_peak], delta_peak, 'ro', label=f'Peak: {delta_peak:.4f} m')
plt.axhline(delta_ss, color='purple', linestyle=':', alpha=0.5, label=f'Steady-State: {delta_ss:.4f} m')
plt.xlabel('Time (s)')
plt.ylabel('Deflection (m)')
plt.title('Coupler Deflection')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Deflection.pdf')
plt.show()

print(f'Peak Deflection (delta_peak): {delta_peak:.4f} m')
print(f'Steady-State Deflection (delta_ss): {delta_ss:.4f} m')
print(f'New Steady-State Velocity (v_ss): {v_ss:.4f} m/s ({v_ss * 3.6:.2f} km/h)')
